# 01 - 마스킹 확산의 기초

**학습 목표**: OCR 정답 토큰을 무작위로 가리고, 확신도가 높은 위치를 한 번에 공개하는 과정을 이해합니다.

**실행 방법**: Python 3/Jupyter에서 cell을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 라이브러리 `random`만 사용합니다.

이 코드는 논문 모델/성능 재현이 아닌 toy reproduction입니다.

In [ ]:
import random
# seed가 있는 난수 생성기를 별도로 두어 실행할 때마다 같은 masking을 재현합니다.

MASK = '[M]'
target = list('OCR IS RIGID')
rng = random.Random(7)

def corrupt(tokens, mask_rate, rng):
    # 각 위치를 독립적으로 가리는 것이 식 (2)의 핵심 직관입니다.
    return [MASK if rng.random() < mask_rate else token for token in tokens]

def reveal_parallel(canvas, truth, threshold):
    revealed = []
    for i, token in enumerate(canvas):
        if token != MASK:
            continue
        # 실제 모델 확률 대신, 문자 종류와 위치로 만든 결정적 toy confidence입니다.
        confidence = 0.995 - 0.018 * (i % 4)
        if confidence >= threshold:
            canvas[i] = truth[i]
            revealed.append(i)
    return revealed

canvas = corrupt(target, mask_rate=0.75, rng=rng)
print('corrupted:', ' '.join(canvas))
for threshold in (0.99, 0.96, 0.0):
    positions = reveal_parallel(canvas, target, threshold)
    print(f'threshold={threshold:.2f}, parallel={positions}:', ' '.join(canvas))

assert canvas == target
print('복원 완료:', ''.join(canvas))

## 관찰

낮은 threshold일수록 한 step에서 더 많은 위치를 확정합니다. 실제 DODO는 틀린 토큰을 되돌리지 않는 carry-over unmasking을 쓰므로, 기본 실험에서는 높은 `p=0.99`를 택합니다.